# Ch 09. 최적화 — 해설

> 이 노트북은 다섯 연습문제의 엄밀한 해설과 재현 가능한 검증을 포함합니다.

## 문제 1

**문제**: SGD, Momentum, Adam을 Rosenbrock 함수에 적용하고 수렴 궤적을 비교하라.

### 엄밀한 해설

통제 변수: **optimizer: SGD, Momentum, Adam**.  측정 지표: **final Rosenbrock objective and distance to optimum**.  해석과 한계: Each optimizer starts at the same point and receives the exact analytic gradient for the same step budget. Learning rates are named protocol choices, so trajectory rankings are not claimed universal.


In [ ]:
import torch
def fg(x):
 a,b=x; return (1-a)**2+100*(b-a*a)**2, torch.stack((-2*(1-a)-400*a*(b-a*a),200*(b-a*a)))
for name in ('sgd','momentum','adam'):
 x=torch.tensor([-1.2,1.]); m=torch.zeros(2); v=torch.zeros(2)
 for t in range(1,2001):
  loss,g=fg(x); lr=1e-3 if name!='adam' else 2e-3
  if name=='sgd': step=g
  elif name=='momentum': m=.9*m+g; step=m
  else: m=.9*m+.1*g; v=.999*v+.001*g*g; step=(m/(1-.9**t))/(torch.sqrt(v/(1-.999**t))+1e-8)
  x=x-lr*step
 print({'optimizer':name,'loss':loss.item(),'distance':torch.linalg.vector_norm(x-torch.ones(2)).item()})


## 문제 2

**문제**: Adam의 편향 보정이 왜 필요한지 early step에서 $\mathbf{m}_t$의 크기를 비교하여 보이라.

### 엄밀한 해설

통제 변수: **Adam first-moment correction: raw versus bias-corrected**.  측정 지표: **moment estimate during first five steps**.  해석과 한계: For constant gradient g, m_t=(1-beta^t)g; division by 1-beta^t removes initialization bias exactly.


In [ ]:
import torch
g=2.; beta=.9; m=0.
for t in range(1,6): m=beta*m+(1-beta)*g; corrected=m/(1-beta**t); print({'step':t,'raw_m':m,'corrected_m':corrected}); assert abs(corrected-g)<1e-12


## 문제 3

**문제**: Adam vs AdamW를 가중치 감쇠 0.01로 설정하여 MNIST에서 비교하라.

### 엄밀한 해설

통제 변수: **regularization rule: Adam L2 coupling versus AdamW decoupling**.  측정 지표: **parameter value after five zero-loss-gradient steps**.  해석과 한계: With zero data gradient, coupled Adam normalizes the L2 term through moments, whereas AdamW multiplies weights by 1-lr*wd. This isolates the mathematical difference, not MNIST accuracy.


In [ ]:
import torch
for name in ('adam_l2','adamw'):
 w=torch.tensor(1.); m=v=0.; lr=.1; wd=.01
 for t in range(1,6):
  if name=='adam_l2': g=wd*w; m=.9*m+.1*g; v=.999*v+.001*g*g; w-=lr*(m/(1-.9**t))/(v/(1-.999**t))**.5
  else: w*=1-lr*wd
 print({'rule':name,'weight_after_5_steps':w.item()})


## 문제 4

**문제**: 학습률 $10^{-2}, 10^{-3}, 10^{-4}$로 Adam을 실행하고 수렴 속도를 비교하라.

### 엄밀한 해설

통제 변수: **Adam learning rate: 1e-2, 1e-3, 1e-4**.  측정 지표: **quadratic loss after 100 matched steps**.  해석과 한계: The objective, start, beta values, and steps are fixed; only learning rate changes. The result describes this finite budget, not an unconditional best rate.


In [ ]:
import torch
for lr in (1e-2,1e-3,1e-4):
 x=torch.tensor(1.); m=v=0.
 for t in range(1,101): g=2*x; m=.9*m+.1*g; v=.999*v+.001*g*g; x-=lr*(m/(1-.9**t))/(torch.sqrt(v/(1-.999**t))+1e-8)
 print({'learning_rate':lr,'loss_after_100':x.square().item()})


## 문제 5

**문제**: Warmup 단계가 없을 때 LLM 학습이 불안정해지는 이유를 설명하라.

### 엄밀한 해설

통제 변수: **schedule: immediate full rate versus linear warmup**.  측정 지표: **maximum parameter update over the first ten steps**.  해석과 한계: Early adaptive statistics and large model gradients can make immediate full-rate updates abrupt. Warmup bounds early update magnitude; this deterministic sequence demonstrates that mechanism.


In [ ]:
import torch
grads=torch.tensor([20.,12.,8.,5.,3.,2.,1.,1.,1.,1.])
for mode in ('no_warmup','linear_warmup'):
 w=torch.tensor(0.); updates=[]
 for t,g in enumerate(grads,1): lr=.1 if mode=='no_warmup' else .1*min(1,t/5); u=lr*g; w-=u; updates.append(abs(float(u)))
 print({'schedule':mode,'max_early_update':max(updates),'parameter':w.item()})
